In [1]:
import pandas as pd
import random

# Categories and tags
categories = {
    "Data Science": ["python", "ml", "statistics", "pandas", "deep learning"],
    "Web Development": ["html", "css", "javascript", "react", "node"],
    "Cloud Computing": ["aws", "azure", "gcp", "docker", "kubernetes"],
    "Cybersecurity": ["network security", "ethical hacking", "cryptography", "firewall", "penetration testing"],
    "Mobile Development": ["android", "ios", "flutter", "react native", "kotlin"],
    "DevOps": ["ci/cd", "jenkins", "docker", "terraform", "linux"]
}

difficulties = ["Beginner", "Intermediate", "Advanced"]

courses = []
course_id = 1

for category, tags in categories.items():
    for difficulty in difficulties:
        for i in range(8):  # 8 courses per difficulty per category
            course = {
                "course_id": course_id,
                "title": f"{category} {difficulty} Course {i+1}",
                "category": category,
                "difficulty": difficulty,
                "tags": ", ".join(random.sample(tags, 3))
            }
            courses.append(course)
            course_id += 1

courses_df = pd.DataFrame(courses)

courses_df.head()

,course_id,title,category,difficulty,tags
0,1,Data Science Beginner Course 1,Data Science,Beginner,"deep learning, ml, python"
1,2,Data Science Beginner Course 2,Data Science,Beginner,"statistics, deep learning, python"
2,3,Data Science Beginner Course 3,Data Science,Beginner,"pandas, ml, statistics"
3,4,Data Science Beginner Course 4,Data Science,Beginner,"pandas, ml, python"
4,5,Data Science Beginner Course 5,Data Science,Beginner,"statistics, python, ml"


In [2]:
# courses_df.to_csv("courses.csv", index=False)

In [3]:
num_users = 500

interests = list(categories.keys())
skill_levels = ["Beginner", "Intermediate", "Advanced"]

users = []

for user_id in range(1, num_users + 1):
    user = {
        "user_id": user_id,
        "interest": random.choice(interests),
        "skill_level": random.choice(skill_levels)
    }
    users.append(user)

users_df = pd.DataFrame(users)

users_df.head()

,user_id,interest,skill_level
0,1,Cybersecurity,Advanced
1,2,Cybersecurity,Intermediate
2,3,Data Science,Intermediate
3,4,Cybersecurity,Intermediate
4,5,DevOps,Beginner


In [4]:
# users_df.to_csv("users.csv", index=False)

In [5]:
ratings = []

for _, user in users_df.iterrows():
    
    # Each user rates 20 random courses
    rated_courses = courses_df.sample(20)

    for _, course in rated_courses.iterrows():

        rating = random.randint(1, 5)

        # Boost ratings if category matches interest
        if course["category"] == user["interest"]:
            rating = random.randint(4, 5)

        # Lower ratings if difficulty mismatch
        if user["skill_level"] == "Beginner" and course["difficulty"] == "Advanced":
            rating = random.randint(1, 2)

        ratings.append({
            "user_id": user["user_id"],
            "course_id": course["course_id"],
            "rating": rating
        })

ratings_df = pd.DataFrame(ratings)

ratings_df.head()

,user_id,course_id,rating
0,1,82,4
1,1,35,3
2,1,100,1
3,1,108,5
4,1,107,2


In [6]:
# ratings_df.to_csv("ratings.csv", index=False)

## Collaborative Filtering

In [7]:
user_item_matrix = ratings_df.pivot_table(
    index='user_id',
    columns='course_id',
    values='rating'
)

user_item_matrix.head()

course_id,1,2,3,4,5,6,7,8,9,10,...,135,136,137,138,139,140,141,142,143,144
user_id,,,,,,,,,,,,,,,,,,,,,
1,2.0,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,2.0,NaN,NaN,NaN,1.0,NaN,NaN,...,NaN,4.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN
3,4.0,NaN,NaN,NaN,5.0,NaN,NaN,5.0,4.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,5.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN
5,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,4.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# Fill missing ratings with 0
matrix_filled = user_item_matrix.fillna(0)

# Calculate user-user similarity
user_similarity = cosine_similarity(matrix_filled)

print(user_similarity.shape)

(500, 500)


In [9]:
import numpy as np

# Convert similarity matrix into DataFrame
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

def recommend_courses(user_id, top_n=5):

    # Get similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)

    # Remove the same user
    similar_users = similar_users.drop(user_id)

    # Top 5 similar users
    top_users = similar_users.head(5).index

    # Courses already rated by target user
    user_courses = set(
        ratings_df[ratings_df["user_id"] == user_id]["course_id"]
    )

    recommendations = {}

    # Find courses liked by similar users
    for sim_user in top_users:

        sim_user_ratings = ratings_df[
            ratings_df["user_id"] == sim_user
        ]

        for _, row in sim_user_ratings.iterrows():

            course_id = row["course_id"]

            # Recommend only unseen courses
            if course_id not in user_courses:

                recommendations[course_id] = (
                    recommendations.get(course_id, 0)
                    + row["rating"]
                )

    # Sort recommendations
    recommended_courses = sorted(
        recommendations.items(),
        key=lambda x: x[1],
        reverse=True
    )

    final_recommendations = []

    for course_id, score in recommended_courses[:top_n]:
    
        course_info = courses_df[
            courses_df["course_id"] == course_id
        ].iloc[0]
    
        final_recommendations.append({
            "course_id": course_id,
            "title": course_info["title"],
            "category": course_info["category"],
            "difficulty": course_info["difficulty"],
            "score": score
        })
    
    return pd.DataFrame(final_recommendations)

In [10]:
recommend_courses(user_id=10, top_n=5)

,course_id,title,category,difficulty,score
0,21,Data Science Advanced Course 5,Data Science,Advanced,6
1,124,DevOps Beginner Course 4,DevOps,Beginner,6
2,55,Cloud Computing Beginner Course 7,Cloud Computing,Beginner,5
3,26,Web Development Beginner Course 2,Web Development,Beginner,5
4,64,Cloud Computing Intermediate Course 8,Cloud Computing,Intermediate,5


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine important text features
courses_df["combined_features"] = (
    courses_df["category"] + " "
    + courses_df["difficulty"] + " "
    + courses_df["tags"]
)

# Convert text into TF-IDF vectors
tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(
    courses_df["combined_features"]
)

print(tfidf_matrix.shape)

(144, 46)


In [12]:
course_similarity = cosine_similarity(tfidf_matrix)

print(course_similarity.shape)

(144, 144)


In [13]:
course_similarity_df = pd.DataFrame(
    course_similarity,
    index=courses_df["course_id"],
    columns=courses_df["course_id"]
)

def content_based_recommend(course_id, top_n=5):

    # Get similarity scores
    similar_courses = course_similarity_df[course_id].sort_values(
        ascending=False
    )

    # Remove same course
    similar_courses = similar_courses.drop(course_id)

    # Top recommendations
    top_courses = similar_courses.head(top_n)

    recommendations = []

    for cid, score in top_courses.items():

        course_info = courses_df[
            courses_df["course_id"] == cid
        ].iloc[0]

        recommendations.append({
            "course_id": cid,
            "title": course_info["title"],
            "category": course_info["category"],
            "difficulty": course_info["difficulty"],
            "similarity_score": round(score, 3)
        })

    return pd.DataFrame(recommendations)

In [14]:
content_based_recommend(course_id=10)

,course_id,title,category,difficulty,similarity_score
0,17,Data Science Advanced Course 1,Data Science,Advanced,0.911
1,7,Data Science Beginner Course 7,Data Science,Beginner,0.911
2,8,Data Science Beginner Course 8,Data Science,Beginner,0.911
3,12,Data Science Intermediate Course 4,Data Science,Intermediate,0.791
4,11,Data Science Intermediate Course 3,Data Science,Intermediate,0.774


In [16]:
def hybrid_recommendation(user_id, top_n=5):

    # Step 1: Collaborative recommendations
    collab_recs = recommend_courses(user_id, top_n=10)

    # Step 2: Find user's highly rated courses
    liked_courses = ratings_df[
        (ratings_df["user_id"] == user_id) &
        (ratings_df["rating"] >= 4)
    ]["course_id"].tolist()

    content_scores = {}

    # Step 3: Get content-based similar courses
    for course_id in liked_courses:

        similar_courses = course_similarity_df[
            course_id
        ].sort_values(ascending=False)

        for sim_course, score in similar_courses.items():

            if sim_course != course_id:

                content_scores[sim_course] = (
                    content_scores.get(sim_course, 0)
                    + score
                )

    # Step 4: Combine scores
    hybrid_scores = {}

    for _, row in collab_recs.iterrows():

        cid = row["course_id"]

        collab_score = row["score"]

        content_score = content_scores.get(cid, 0)

        final_score = (
            0.7 * collab_score
            + 0.3 * content_score
        )

        hybrid_scores[cid] = final_score

    # Step 5: Sort final recommendations
    sorted_recommendations = sorted(
        hybrid_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for cid, score in sorted_recommendations[:top_n]:

        course_info = courses_df[
            courses_df["course_id"] == cid
        ].iloc[0]

        recommendations.append({
            "course_id": cid,
            "title": course_info["title"],
            "category": course_info["category"],
            "difficulty": course_info["difficulty"],
            "hybrid_score": round(score, 2)
        })

    return pd.DataFrame(recommendations)

In [17]:
hybrid_recommendation(user_id=10, top_n=5)

,course_id,title,category,difficulty,hybrid_score
0,124,DevOps Beginner Course 4,DevOps,Beginner,4.50
1,21,Data Science Advanced Course 5,Data Science,Advanced,4.50
2,73,Cybersecurity Beginner Course 1,Cybersecurity,Beginner,4.37
3,64,Cloud Computing Intermediate Course 8,Cloud Computing,Intermediate,4.12
4,91,Cybersecurity Advanced Course 3,Cybersecurity,Advanced,4.10


In [27]:
train_data = []
test_data = []

for user_id in ratings_df["user_id"].unique():

    user_ratings = ratings_df[
        ratings_df["user_id"] == user_id
    ]

    # Only liked courses
    liked_courses = user_ratings[
        user_ratings["rating"] >= 4
    ]

    # Skip users with very few liked courses
    if len(liked_courses) < 2:
        continue

    # Randomly select one liked course for testing
    test_sample = liked_courses.sample(1)

    # Remaining ratings go to training
    train_sample = user_ratings.drop(test_sample.index)

    train_data.append(train_sample)
    test_data.append(test_sample)

# Combine all users' data
train_df = pd.concat(train_data)
test_df = pd.concat(test_data)

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

Train Shape: (9500, 3)
Test Shape: (500, 3)


In [28]:
train_matrix = train_df.pivot_table(
    index='user_id',
    columns='course_id',
    values='rating'
)

train_matrix_filled = train_matrix.fillna(0)

train_user_similarity = cosine_similarity(
    train_matrix_filled
)

train_user_similarity_df = pd.DataFrame(
    train_user_similarity,
    index=train_matrix.index,
    columns=train_matrix.index
)

In [29]:
def train_recommend_courses(user_id, top_n=5):

    similar_users = train_user_similarity_df[user_id]\
        .sort_values(ascending=False)

    similar_users = similar_users.drop(user_id)

    top_users = similar_users.head(5).index

    user_courses = set(
        train_df[
            train_df["user_id"] == user_id
        ]["course_id"]
    )

    recommendations = {}

    for sim_user in top_users:

        sim_user_ratings = train_df[
            train_df["user_id"] == sim_user
        ]

        for _, row in sim_user_ratings.iterrows():

            course_id = row["course_id"]

            if course_id not in user_courses:

                recommendations[course_id] = (
                    recommendations.get(course_id, 0)
                    + row["rating"]
                )

    recommended_courses = sorted(
        recommendations.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [cid for cid, _ in recommended_courses[:top_n]]

In [30]:
def evaluate_precision_at_k(k=5):

    hits = 0
    total_users = 0

    for _, row in test_df.iterrows():

        user_id = row["user_id"]
        hidden_course = row["course_id"]

        # Get recommendations from training data
        recommendations = train_recommend_courses(
            user_id,
            top_n=k
        )

        # Check if hidden course is recommended
        if hidden_course in recommendations:
            hits += 1

        total_users += 1

    precision = hits / (total_users * k)

    recall = hits / total_users

    return {
        "Precision@K": round(precision, 4),
        "Recall@K": round(recall, 4),
        "Hits": hits,
        "Total Users": total_users
    }

In [31]:
evaluate_precision_at_k(k=5)

{'Precision@K': 0.0068, 'Recall@K': 0.034, 'Hits': 17, 'Total Users': 500}